In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import bct
import numpy as np
import plotly.graph_objects as go
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from tvb.datatypes.connectivity import Connectivity
from tvbwidgets.ui.head_widget import HeadWidget
from tvbwidgets.ui.connectivity_matrix_editor_widget import ConnectivityMatrixEditor
import k3d

class BCTConnectivityMatrixEditor(ConnectivityMatrixEditor):
    def __init__(self, connectivity, **kwargs):
        super().__init__(connectivity, **kwargs)
        self.tab.children = [self.tab.children[0]]
        self.tab.set_title(0, "weights")

class ColoredConnectivityHeadWidget(HeadWidget):

    def __init__(self, datatypes, node_values=None, region_labels=None, **kwargs):
        self.node_values = node_values
        self.region_labels = region_labels
        self._info_label = widgets.HTML(
            value="<i style='color:gray; font-size:13px'>Click a node to see its value</i>",
            layout=widgets.Layout(margin="6px 0 0 0")
        )
        super().__init__(datatypes, **kwargs)

    def _HeadWidget__draw_connectivity(self, connectivity):
        self._centres = connectivity.centres
        self._labels = self.region_labels if self.region_labels is not None else connectivity.region_labels

        values = self.node_values
        if values is None:
            values = np.ones(len(connectivity.centres))
        values = np.array(values, dtype=float)
        values[np.isinf(values)] = 0
        self._values = values

        self._points_obj = k3d.points(
            connectivity.centres,
            point_size=8,
            shader='dot',
            attribute=values,
            color_map=k3d.matplotlib_color_maps.viridis,
            color_range=[values.min(), values.max()],
        )

        edge_indices = np.nonzero(connectivity.weights)
        edges = list(zip(edge_indices[0], edge_indices[1]))

        lines = k3d.lines(
            connectivity.centres,
            indices=edges,
            shader='simple',
            color=0xffffff,
            width=2,
        )

        self.plot += self._points_obj
        self.plot += lines

    def display(self):
        from IPython.display import display as ipy_display
        ipy_display(self._info_label)

def plot_bct_histogram(region_labels, node_values, analyzer_name):
    values = np.array(node_values, dtype=float)
    values[np.isinf(values)] = 0

    norm = mcolors.Normalize(vmin=values.min(), vmax=values.max())
    cmap = cm.get_cmap("plasma")
    bar_colors = [
        f"rgba({int(r*255)},{int(g*255)},{int(b*255)},{a:.2f})"
        for r, g, b, a in [cmap(norm(v)) for v in values]
    ]

    fig = go.Figure(go.Bar(
        x=list(region_labels),
        y=values.tolist(),
        marker=dict(color=bar_colors),
        customdata=np.stack([region_labels, values], axis=1),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            f"{analyzer_name}: <b>%{{customdata[1]:.4f}}</b><br>"
            "<extra></extra>"
        ),
        text=[f"{v:.3f}" for v in values],
        textposition="outside",
        textfont=dict(size=9, color="white"),
    ))

    fig.update_layout(
        title=dict(
            text=f"BCT — {analyzer_name}",
            font=dict(color="white", size=14),
        ),
        paper_bgcolor="#2b2b2b",
        plot_bgcolor="#3a3a3a",
        xaxis=dict(
            tickangle=-90,
            tickfont=dict(size=8, color="white"),
            gridcolor="#555",
            linecolor="#555",
        ),
        yaxis=dict(
            title=analyzer_name,
            titlefont=dict(color="white"),
            tickfont=dict(color="white"),
            gridcolor="#555",
            linecolor="#555",
        ),
        hoverlabel=dict(
            bgcolor="#1a1a2e",
            font_size=12,
            font_color="white",
        ),
        margin=dict(l=60, r=40, t=50, b=160),
        height=550,
        dragmode="zoom",
    )

    fig.update_xaxes(
        rangeslider=dict(visible=True, thickness=0.04, bgcolor="#444"),
    )

    fig.show()

ANALYZERS = {
    "Betweenness centrality": {
        "description": "Fraction of shortest paths passing through each node (binary)",
        "fn": lambda c: bct.betweenness_bin(c.binarized_weights),
    },
    "Clustering coefficient": {
        "description": "Proportion of triangle closure around each node (weighted)",
        "fn": lambda c: bct.clustering_coef_wu(c.weights),
    },
    "Eigenvector centrality": {
        "description": "Influence of a node weighted by neighbour influence (weighted)",
        "fn": lambda c: bct.eigenvector_centrality_und(c.weights),
    },
}


default_connectivity = Connectivity.from_file()


editor_instance = {"editor": None}


title = widgets.HTML("<h3 style='margin: 0 0 8px'>Brain Connectivity Toolbox</h3>")

selector = widgets.ToggleButtons(
    options=list(ANALYZERS.keys()),
    layout=widgets.Layout(width="100%"),
)

desc_label = widgets.HTML(
    value=f"<i style='color:gray'>{ANALYZERS[selector.value]['description']}</i>"
)

hint = widgets.HTML(
    "<span style='color:#888; font-size:12px'>Edit connectivity values below, then hit Save before running.</span>"
)

editor_output = widgets.Output()

run_btn = widgets.Button(
    description="Run analysis",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="160px", margin="8px 0"),
)
status = widgets.HTML("")

viz_output  = widgets.Output()
hist_output = widgets.Output()

def show_editor(connectivity):
    with editor_output:
        clear_output(wait=True)
        ed = BCTConnectivityMatrixEditor(connectivity)
        editor_instance["editor"] = ed
        ed.display()

def on_analyzer_change(change):
    desc_label.value = f"<i style='color:gray'>{ANALYZERS[change['new']]['description']}</i>"
    status.value = ""
    show_editor(default_connectivity)
    with viz_output:
        clear_output()
    with hist_output:
        clear_output()

def on_run(btn):
    run_btn.disabled = True
    status.value = "<span style='color:orange'>⏳ Running...</span>"

    with viz_output:
        clear_output(wait=True)
    with hist_output:
        clear_output(wait=True)

    try:
        ed = editor_instance["editor"]
        connectivity = ed.get_connectivity() if ed is not None else default_connectivity

        diff = connectivity.weights - default_connectivity.weights
        changed = np.argwhere(diff != 0)
        with viz_output:
            if len(changed) == 0:
                print("Using default connectivity (no edits detected)")
            else:
                print(f"Using edited connectivity — {len(changed)} cell(s) modified:")
                for row, col in changed:
                    original = default_connectivity.weights[row, col]
                    edited = connectivity.weights[row, col]
                    print(f"  [{connectivity.region_labels[row]}] → [{connectivity.region_labels[col]}]  {original:.4f} → {edited:.4f}")
            print("─" * 40)

        node_values = ANALYZERS[selector.value]["fn"](connectivity)

       
        with viz_output:
            viz = ColoredConnectivityHeadWidget(
                [connectivity],
                node_values=node_values,
                region_labels=connectivity.region_labels,
            )
            viz._current_analyzer = selector.value
            viz.display()
            display(viz)

      
        with hist_output:
            plot_bct_histogram(
                region_labels=connectivity.region_labels,
                node_values=node_values,
                analyzer_name=selector.value,
            )

        status.value = f"<span style='color:green'>✓ {selector.value} done</span>"

    except Exception as e:
        status.value = f"<span style='color:red'>✗ Error: {e}</span>"
    finally:
        run_btn.disabled = False

selector.observe(on_analyzer_change, names="value")
run_btn.on_click(on_run)

with editor_output:
    display(widgets.HTML(
        "<i style='color:gray; font-size:13px'>Select an analyzer above to load the connectivity editor.</i>"
    ))

divider = widgets.HTML("<hr style='margin: 12px 0; border-color: #ddd'>")

viz_tabs = widgets.Tab()
viz_tabs.children = [viz_output, hist_output]
viz_tabs.set_title(0, "3D Head View")
viz_tabs.set_title(1, "Histogram")

ui = widgets.VBox([
    title,
    selector,
    desc_label,
    divider,
    hint,
    editor_output,
    divider,
    widgets.HBox([run_btn, status]),
    viz_tabs,
], layout=widgets.Layout(padding="12px", gap="4px"))

display(ui)

23-03-2026 09:46:15 - INFO - tvbwidgets - Version: 2.3.2
2026-03-23 09:46:17,147 - WARNING - tvb.basic.readers - File 'hemispheres' not found in ZIP.


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\deept\.pyenv\pyenv-win\versions\3.10.11\lib\logging\handlers.py", line 74, in emit
    self.doRollover()
  File "C:\Users\deept\.pyenv\pyenv-win\versions\3.10.11\lib\logging\handlers.py", line 435, in doRollover
    self.rotate(self.baseFilename, dfn)
  File "C:\Users\deept\.pyenv\pyenv-win\versions\3.10.11\lib\logging\handlers.py", line 115, in rotate
    os.rename(source, dest)
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\deept\\.tvb-temp\\logs\\library.log' -> 'C:\\Users\\deept\\.tvb-temp\\logs\\library.log.2026-03-22'
Call stack:
  File "C:\Users\deept\.pyenv\pyenv-win\versions\3.10.11\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\deept\.pyenv\pyenv-win\versions\3.10.11\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\deept\.pyenv\pyen